In [1]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [2]:
# Load base ED patient cohort from BigQuery (built by cohort_base.py).
# One row per ED visit. Retains all discharge_location values at this stage,
# including AGAINST ADVICE — those are removed in the next step.

df_cohort = client.query("SELECT * FROM `ads-599-final.rl_project.cohort_base`").to_dataframe()
print(f"Shape: {df_cohort.shape}")
print(f"\n--- Cohort Label Distribution ---")
print(df_cohort['cohort_label'].value_counts())
df_cohort.head()

Shape: (399573, 24)

--- Cohort Label Distribution ---
cohort_label
ED_DISCHARGE_STABLE        241206
ED_WARD_DISCHARGE          126507
ED_DIRECT_ICU               22993
ED_WARD_ICU                  8491
ED_DISCHARGE_RETURN_ICU       304
ED_DISCHARGE_DIED_72H          72
Name: count, dtype: int64


,subject_id,ed_stay_id,hadm_id,ed_intime,ed_outtime,disposition,race,arrival_transport,first_careunit,first_icu_intime,...,anchor_year,age_at_visit,dod,admittime,dischtime,admission_type,discharge_location,insurance,language,marital_status
0,10002013,34931809,21763296,2165-11-22 20:54:00,2165-11-23 08:20:42,ADMITTED,WHITE,WALK IN,Medical Intensive Care Unit (MICU),2165-11-23 08:20:42,...,2156,62,NaT,2165-11-23 08:19:00,2165-11-26 15:40:00,DIRECT EMER.,HOME HEALTH CARE,Medicaid,English,SINGLE
1,10014354,38849383,27562275,2148-07-17 20:36:00,2148-07-18 04:46:00,ADMITTED,WHITE,AMBULANCE,Cardiology Surgery Intermediate,NaT,...,2146,62,2153-04-07,2148-07-18 02:31:00,2148-07-20 17:47:00,DIRECT EMER.,HOME HEALTH CARE,Medicare,English,SINGLE
2,10018862,35211473,22949345,2149-05-01 17:05:00,2149-05-01 22:22:00,ADMITTED,WHITE,WALK IN,Transplant,NaT,...,2148,57,NaT,2149-05-01 21:43:00,2149-05-05 18:20:00,DIRECT EMER.,HOME,Medicaid,English,MARRIED
3,10019003,38020791,26529390,2155-05-17 21:03:00,2155-05-18 00:03:15,ADMITTED,WHITE,WALK IN,Hematology/Oncology Intermediate,NaT,...,2148,72,2155-12-03,2155-05-17 00:02:00,2155-05-19 18:27:00,DIRECT EMER.,HOME,Medicare,English,WIDOWED
4,10021395,31017572,21372240,2132-07-30 18:58:00,2132-07-31 04:56:00,ADMITTED,WHITE - OTHER EUROPEAN,WALK IN,Medicine,NaT,...,2128,90,NaT,2132-07-31 03:28:00,2132-07-31 15:09:00,DIRECT EMER.,HOME,Medicare,English,WIDOWED


In [3]:
# Remove AGAINST ADVICE discharges.
# These represent patient-driven departures (left AMA, eloped) rather than
# provider disposition decisions, so they're out of scope for the RL policy.
advice_df = df_cohort[df_cohort['discharge_location'] != 'AGAINST ADVICE'].copy()
print(f"Dropped {len(df_cohort) - len(advice_df):,} AGAINST ADVICE rows")
print(f"Remaining: {len(advice_df):,}")

Dropped 1,563 AGAINST ADVICE rows
Remaining: 398,010


In [4]:
# Collapse consecutive ED visits that share a hadm_id into a single row.
# Some patients have two back-to-back ED stays under one admission (e.g. brief
# discharge and immediate return). We keep the first stay's identifiers and
# merge the second stay's ed_outtime/disposition/cohort_label into it,
# storing the second ed_stay_id in ed_stay_id_2.
dup_counts = advice_df['hadm_id'].value_counts()
dup_hadm_ids = dup_counts[dup_counts == 2].index

singles = advice_df[~advice_df['hadm_id'].isin(dup_hadm_ids)].copy()
dups = advice_df[advice_df['hadm_id'].isin(dup_hadm_ids)].sort_values(['hadm_id', 'ed_intime'])

first_rows = dups.groupby('hadm_id').nth(0).reset_index(drop=True)
second_rows = dups.groupby('hadm_id').nth(1).reset_index(drop=True)

merged_dups = first_rows.copy()
merged_dups['ed_outtime'] = second_rows['ed_outtime'].values
merged_dups['disposition'] = second_rows['disposition'].values
merged_dups['cohort_label'] = second_rows['cohort_label'].values
merged_dups['ed_stay_id_2'] = second_rows['ed_stay_id'].values
merged_dups.drop(columns=['base_pathway'], inplace=True)

singles['ed_stay_id_2'] = float('nan')
singles.drop(columns=['base_pathway'], inplace=True)

cohort_df = pd.concat([singles, merged_dups], ignore_index=True)

print(f"advice_df rows: {len(advice_df):,}")
print(f"cohort_df rows: {len(cohort_df):,}  (expected {len(advice_df) - len(dup_hadm_ids):,})")
print(f"Max hadm_id count: {cohort_df['hadm_id'].value_counts().max()}  (expected 1)")

advice_df rows: 398,010
cohort_df rows: 397,675  (expected 397,675)
Max hadm_id count: 1  (expected 1)


In [5]:
# Add stay window columns: full time span from ED arrival to final discharge.
# For admitted patients: ed_intime → dischtime
# For ED-only patients: ed_intime → ed_outtime
cohort_df['stay_window_start'] = pd.to_datetime(cohort_df['ed_intime'])
cohort_df['stay_window_end'] = pd.to_datetime(
    cohort_df['dischtime'].fillna(cohort_df['ed_outtime'])
)

# Add ED boarding time: hours spent in ED after admission decision was made.
# Calculated as ed_outtime - admittime (null for ED-only discharges).
cohort_df['ed_boarding_time_hrs'] = (
    (pd.to_datetime(cohort_df['ed_outtime']) - pd.to_datetime(cohort_df['admittime']))
    .dt.total_seconds()
    / 3600
)

print(f"Patients with boarding time (admitted):   {cohort_df['ed_boarding_time_hrs'].notna().sum():,}")
print(f"Patients without boarding time (ED-only): {cohort_df['ed_boarding_time_hrs'].isna().sum():,}")

Patients with boarding time (admitted):   192,201
Patients without boarding time (ED-only): 205,474


In [6]:
PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

info = DatasetInfo(
    description=(
        "Processed ED patient cohort derived from MIMIC-IV. One row per ED visit (ed_stay_id). "
        "Built from cohort_base (BigQuery) with the following post-processing steps: "
        "(1) AGAINST ADVICE discharge_location records removed — patient-driven departures are out of scope. "
        "(2) Consecutive ED visits sharing a hadm_id collapsed into single rows; second ed_stay_id stored in ed_stay_id_2. "
        "(3) stay_window_start/stay_window_end columns added covering ED arrival to final discharge. "
        "(4) ed_boarding_time_hrs added: hours in ED after admission decision (null for ED-only patients). "
        "Intended as the primary cohort table for feature engineering and RL state construction."
    ),
    license=PHYSIONET_LICENSE,
)

ds = Dataset.from_pandas(cohort_df, split='cohort_full', info=info)
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="cohort_full", data_dir="cohort")
print("Pushed cohort to HuggingFace Hub.")

Setting num_proc from 1 back to 1 for the cohort_full split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Pushed cohort to HuggingFace Hub.


In [10]:
import pickle

stay_id_remap = (
    cohort_df[cohort_df['ed_stay_id_2'].notna()]
    .set_index('ed_stay_id_2')['ed_stay_id']
    .astype(int)
    .to_dict()
)

with open("stay_id_remap.pkl", "wb") as f:
    pickle.dump(stay_id_remap, f)

print(f"Saved stay_id_remap with {len(stay_id_remap):,} entries to stay_id_remap.pkl")

Saved stay_id_remap with 335 entries to stay_id_remap.pkl
